In [ ]:
import sympy as sp
from sympy import *
from IPython.display import display, Math

def print_math(str):
    display(Math(str))

Q1: Derive jacobian matrix.

In [38]:
# Symbols
rho, u, E = sp.symbols('rho u E')
gamma = sp.symbols('gamma')
U1, U2, U3 = sp.symbols('U1 U2 U3')
U_sym = sp.Matrix([U1, U2, U3])

# Pressure
p = (gamma - 1) * (U3 - sp.Rational(1,2) * U2**2 / U1)
# Flux components
F1 = U2
F2 = U1 * (U2/U1)**2 + p
F3 = (U2/U1) * (U3 + p)

# Flux vector
F = sp.Matrix([F1, F2, F3])
F

Matrix([
[                                          U2],
[  (U3 - U2**2/(2*U1))*(gamma - 1) + U2**2/U1],
[U2*(U3 + (U3 - U2**2/(2*U1))*(gamma - 1))/U1]])

In [43]:
A=F.jacobian(U_sym)
A


Matrix([
[                                                                             0,                                                                   1,           0],
[                                     U2**2*(gamma - 1)/(2*U1**2) - U2**2/U1**2,                                        -U2*(gamma - 1)/U1 + 2*U2/U1,   gamma - 1],
[-U2*(U3 + (U3 - U2**2/(2*U1))*(gamma - 1))/U1**2 + U2**3*(gamma - 1)/(2*U1**3), (U3 + (U3 - U2**2/(2*U1))*(gamma - 1))/U1 - U2**2*(gamma - 1)/U1**2, U2*gamma/U1]])

In [ ]:
U_subs = {
    U1: rho,
    U2: rho*u,
    U3: rho*E
}

cv, T, a, R = sp.symbols('cv T a R')
thermo_subs = {
    E:cv*T + u**2/2,
    cv:R/(gamma - 1),
    T: a**2 / (gamma * R)
}

A = A.subs(U_subs).subs(thermo_subs).simplify()
A


Matrix([
[                                                                  0,                                                                         1,         0],
[                                                 u**2*(gamma - 3)/2,                                                             u*(3 - gamma), gamma - 1],
[u*(-2*a**2 + gamma**2*u**2 - 3*gamma*u**2 + 2*u**2)/(2*(gamma - 1)), (2*a**2 + gamma*u**2*(gamma - 1) - 3*u**2*(gamma - 1)**2)/(2*(gamma - 1)),   gamma*u]])

In [42]:
U =  U_sym.subs(U_subs).subs(thermo_subs)
print_math("U =")
U

<IPython.core.display.Math object>

Matrix([
[                                    rho],
[                                  rho*u],
[rho*(a**2/(gamma*(gamma - 1)) + u**2/2)]])

Q2: Derive right eigenvectors. QA = [r1 r2 r3]

In [ ]:
eigenvalues = [val for val, _, _ in A.eigenvects()]
eigenvectors = [v for _, _, vs in A.eigenvects() for v in vs]

# rescaling and reording eigenvectors to be consistent with notes
r1= eigenvectors[0]*(1/eigenvectors[0][0])
r2= eigenvectors[2]*(1/eigenvectors[2][0]) 
r3= -eigenvectors[1]*(1/eigenvectors[1][0])

print("Expanded, unscaled by rho/(2a)")
print_math("Q_A=")
Matrix.hstack(r1, r2, r3)


Expanded, unscaled by rho/(2a)


<IPython.core.display.Math object>

Matrix([
[     1,                                                                1,                                                                -1],
[     u,                (2*a*gamma - 2*a + 2*gamma*u - 2*u)/(2*gamma - 2),               -(-2*a*gamma + 2*a + 2*gamma*u - 2*u)/(2*gamma - 2)],
[u**2/2, (2*a**2 + 2*a*gamma*u - 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2), -(2*a**2 - 2*a*gamma*u + 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2)]])

In [150]:
# collect and simplify entries of r1, r2, r3 
def clean_r(v):
    return Matrix([collect((simplify(entry)), [u**2/2,  a**2,u*a]) for entry in v])
# scale r2 and r3 by rho/(2a) to be consistent with notes
r1=clean_r(r1)
r2=clean_r(r2)*rho/(2*a)
r3=clean_r(r3)*rho/(2*a)
Qa = Matrix.hstack(r1, r2, r3)

print_math("Q_A=")
Qa

<IPython.core.display.Math object>

Matrix([
[     1,                                                           rho/(2*a),                                                           -rho/(2*a)],
[     u,                                                   rho*(a + u)/(2*a),                                                    rho*(a - u)/(2*a)],
[u**2/2, rho*(a**2 + a*u*(gamma - 1) + u**2*(gamma - 1)/2)/(2*a*(gamma - 1)), rho*(-a**2 + a*u*(gamma - 1) + u**2*(1 - gamma)/2)/(2*a*(gamma - 1))]])

Q3: Derive flux-vector splitting

In [ ]:
l1, l2, l3 = sp.symbols('l1 l2 l3')

Lambda = sp.diag(l1, l2, l3)
lambda_subs = {
    l1: eigenvalues[0],
    l2: eigenvalues[1],
    l3: eigenvalues[2]
}

print_math("\\Lambda =")
Lambda.subs(lambda_subs)

<IPython.core.display.Math object>

Matrix([
[u,      0,     0],
[0, -a + u,     0],
[0,      0, a + u]])

In [ ]:
Qa_inv = Qa.inv()
print_math("Q_A^{-1}=")
Qa_inv

<IPython.core.display.Math object>

Matrix([
[ (2*a**2 - gamma*u**2 + u**2)/(2*a**2),        (gamma*u - u)/a**2,    (1 - gamma)/a**2],
[(-2*a*u + gamma*u**2 - u**2)/(2*a*rho), (a - gamma*u + u)/(a*rho), (gamma - 1)/(a*rho)],
[(-2*a*u - gamma*u**2 + u**2)/(2*a*rho), (a + gamma*u - u)/(a*rho), (1 - gamma)/(a*rho)]])

In [136]:
F_pm =Qa*Lambda*Qa_inv*U
F_pm_factored = Matrix([factor(collect(row, [l1, l2, l3])) for row in F_pm/(rho/(2*gamma))])
F_pm_factored[2] = collect((F_pm_factored[2])*2*(gamma-1), [l1, l2, l3])/(2*(gamma-1))
print_math("F^{\\pm}=\\frac{\\rho}{2\\gamma}" + sp.latex(F_pm_factored)) 


<IPython.core.display.Math object>

In [ ]:
Lambda_QaInv_U= simplify(Lambda*Qa_inv*U) 
r1*Lambda_QaInv_U[0] + r2*Lambda_QaInv_U[1] + r3*Lambda_QaInv_U[2]

Matrix([
[                                                                                                                              l1*rho*(gamma - 1)/gamma + l2*rho/(2*gamma) + l3*rho/(2*gamma)],
[                                                                                                            l1*rho*u*(gamma - 1)/gamma + l2*rho*(a + u)/(2*gamma) - l3*rho*(a - u)/(2*gamma)],
[l1*rho*u**2*(gamma - 1)/(2*gamma) + l2*rho*(a**2 + a*u*(gamma - 1) + u**2*(gamma - 1)/2)/(2*gamma*(gamma - 1)) - l3*rho*(-a**2 + a*u*(gamma - 1) + u**2*(1 - gamma)/2)/(2*gamma*(gamma - 1))]])

In [ ]:
Lambda_QaInv_U= simplify(Lambda*Qa_inv*U) 
print_math("F^{\\pm} = \\frac{1}{\\gamma}Q_A" + sp.latex(Lambda_QaInv_U))

print_math("F^{\\pm} = " + sp.latex(Lambda_QaInv_U[0]) + sp.latex(r1)+ "+" + sp.latex(Lambda_QaInv_U[1]) + sp.latex(r2) + "+" + sp.latex(Lambda_QaInv_U[2]) + sp.latex(r3))

<IPython.core.display.Math object>

<IPython.core.display.Math object>